# Dream Team Framework: Shelf Life Prediction Challenge

This notebook demonstrates the Dream Team framework on the AgentDS Food Production benchmark.

**Challenge:** Predict remaining shelf life in days for food production batches
**Metric:** Mean Absolute Error (MAE)

## Framework Features:
- 🧬 **Evolving Agents**: Start as generalists, evolve into specialists
- 📚 **Research Integration**: Autonomous paper search and insight extraction
- 🔄 **Evolution Triggers**: Automatic detection of when agents need to evolve
- 🎯 **Benchmark-Driven**: Metrics guide agent evolution

## Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent / 'src'))

from dream_team import (
    Agent, 
    TeamMeeting, 
    IndividualMeeting,
    EvolutionEngine, 
    get_research_assistant,
    PerformancePlateauTrigger,
    save_json,
    load_json
)

# Setup paths
DATA_DIR = Path('../data/FoodProduction')
RESULTS_DIR = Path('../results/shelf_life')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Dream Team framework loaded")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Results directory: {RESULTS_DIR}")

## 1. Data Exploration

In [ ]:
# Load training data
batches_train = pd.read_csv(DATA_DIR / 'batches_train.csv')
batches_test = pd.read_csv(DATA_DIR / 'batches_test.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
sites = pd.read_csv(DATA_DIR / 'sites.csv')
regions = pd.read_csv(DATA_DIR / 'regions.csv')

print(f"Training batches: {len(batches_train):,}")
print(f"Test batches: {len(batches_test):,}")
print(f"Products: {len(products):,}")
print(f"Sites: {len(sites):,}")
print(f"Regions: {len(regions):,}")

print("\n📊 Training Data Sample:")
display(batches_train.head())

print("\n📊 Data Info:")
print(batches_train.info())

print("\n📊 Target Distribution:")
print(batches_train['shelf_life_remaining_days'].describe())

In [ ]:
# Join with reference data for richer context
train_enriched = batches_train.merge(products, on='sku_id', how='left')
train_enriched = train_enriched.merge(sites, on='site_id', how='left')

print("✅ Enriched training data with product and site information")
display(train_enriched.head())

## 2. Create Initial Team

We'll start with a small team of generalist agents who will evolve based on the challenge.

In [ ]:
# Create initial team
pi = Agent(
    title="Principal Investigator",
    expertise="data science, machine learning, research strategy, experimental design",
    goal="solve the shelf life prediction challenge with high accuracy (low MAE)",
    role="lead the team, make strategic decisions, coordinate research efforts"
)

data_scientist = Agent(
    title="Data Scientist",
    expertise="exploratory data analysis, feature engineering, statistical modeling, Python",
    goal="understand the data deeply and create informative features",
    role="analyze data patterns, engineer features, propose modeling approaches"
)

ml_engineer = Agent(
    title="ML Engineer",
    expertise="scikit-learn, model training, hyperparameter tuning, cross-validation",
    goal="build and optimize prediction models",
    role="implement models, tune hyperparameters, evaluate performance"
)

team_members = [data_scientist, ml_engineer]

# Save initial team
pi.save(RESULTS_DIR / 'agents' / 'pi_initial.json')
data_scientist.save(RESULTS_DIR / 'agents' / 'data_scientist_initial.json')
ml_engineer.save(RESULTS_DIR / 'agents' / 'ml_engineer_initial.json')

print("✅ Initial team created:")
print(f"  - {pi.title}: {pi.role}")
print(f"  - {data_scientist.title}: {data_scientist.role}")
print(f"  - {ml_engineer.title}: {ml_engineer.role}")

## 3. Initial Team Meeting: Problem Analysis

The team will discuss the challenge and formulate an initial approach.

In [ ]:
# Prepare context for the meeting
problem_context = f"""
Challenge: Shelf Life Prediction for Food Production Batches

Data Overview:
- Training samples: {len(batches_train):,}
- Test samples: {len(batches_test):,}
- Target: shelf_life_remaining_days (continuous)
- Metric: MAE (Mean Absolute Error)

Available Features:
- batch_id: Unique batch identifier
- sku_id: Product SKU (can join with products table)
- site_id: Production site (can join with sites table)
- dwell_hours: Hours in storage
- mean_temp_F: Average temperature in Fahrenheit
- mean_rh_pct: Average relative humidity percentage
- door_opens_count: Number of door openings

Reference Data:
- products.csv: category, storage_class, base_shelf_life_days
- sites.csv: region_id, line_type
- regions.csv: seasonality_amp

Target Statistics:
{batches_train['shelf_life_remaining_days'].describe()}

Key Challenges:
1. Food perishability - limited shelf life
2. Storage conditions impact (temperature, humidity)
3. Door opening patterns (temperature abuse)
4. Different product categories and storage classes
5. Site and regional variations
"""

print("📋 Problem Context Prepared")
print(problem_context)

In [ ]:
# Run initial team meeting
meeting = TeamMeeting(save_dir=str(RESULTS_DIR / 'meetings'))

agenda = f"""
Analyze the shelf life prediction challenge and develop an initial modeling strategy.

Discussion Points:
1. What are the key factors affecting shelf life?
2. What features should we engineer?
3. What modeling approaches should we try?
4. How should we validate our models?
5. What domain knowledge do we need to acquire?

Context:
{problem_context}
"""

print("🏁 Starting initial team meeting...\n")

meeting_summary = meeting.run(
    team_lead=pi,
    team_members=team_members,
    agenda=agenda,
    num_rounds=2
)

print("\n✅ Meeting completed!")
print("\n📝 Meeting Summary:")
print(meeting_summary)

## 4. Research Phase: Literature Review

Agents will research relevant papers to gain domain expertise.

In [ ]:
# Research relevant papers
research = get_research_assistant()

print("🔬 Researching shelf life prediction literature...\n")

papers = research.research_topic(
    query="shelf life prediction food storage temperature humidity",
    context="""Predicting remaining shelf life for food products based on storage conditions.
    Key factors: temperature, humidity, door openings, product category.
    Goal: Minimize prediction error (MAE) for days remaining.""",
    num_papers=5
)

print(f"\n✅ Found {len(papers)} relevant papers\n")

for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper.title}")
    print(f"   Authors: {', '.join(paper.authors[:3])}{'...' if len(paper.authors) > 3 else ''}")
    print(f"   Year: {paper.year}")
    print(f"   Relevance: {paper.relevance_score}/10")
    if paper.key_findings:
        print(f"   Key Finding: {paper.key_findings[0][:100]}...")
    print()

## 5. Agent Evolution: Incorporate Research Insights

Evolve the data scientist with domain knowledge from papers.

In [ ]:
# Evolve data scientist with research insights
evolution_engine = EvolutionEngine()

print("🧬 Evolving Data Scientist with research insights...\n")

evolution_context = {
    "problem_description": "Shelf life prediction for food products",
    "current_challenges": [
        "Understanding temperature abuse impact on shelf life",
        "Modeling humidity effects on perishable goods",
        "Incorporating product-specific deterioration rates"
    ],
    "performance_metrics": "Need to minimize MAE on shelf life predictions"
}

evolved = evolution_engine.evolve_agent(
    agent=data_scientist,
    context=evolution_context,
    papers=papers,
    trigger_reason="Need domain expertise in food science and shelf life modeling"
)

if evolved:
    print("\n✅ Agent evolved successfully!")
    print(f"\nNew Title: {data_scientist.title}")
    print(f"New Expertise: {data_scientist.expertise}")
    print(f"Specialization Depth: {data_scientist.specialization_depth}")
    print(f"\nKnowledge Base Size: {len(data_scientist.knowledge_base.domain_facts)} facts, {len(data_scientist.knowledge_base.papers)} papers")
    
    # Save evolved agent
    data_scientist.save(RESULTS_DIR / 'agents' / 'data_scientist_evolved_v1.json')
else:
    print("\n❌ Evolution not triggered (criteria not met)")

## 6. Feature Engineering Meeting

Now with evolved expertise, the team designs features.

In [ ]:
# Individual work session for feature engineering
individual_meeting = IndividualMeeting(save_dir=str(RESULTS_DIR / 'meetings'))

feature_task = f"""
Design a comprehensive feature engineering strategy for shelf life prediction.

Available Data:
- Batch features: dwell_hours, mean_temp_F, mean_rh_pct, door_opens_count
- Product info: category, storage_class, base_shelf_life_days
- Site info: region_id, line_type
- Region info: seasonality_amp

Consider:
1. Temperature abuse indicators (e.g., thermal load, degree-days above threshold)
2. Humidity stress factors
3. Door opening impact (frequency, disruption)
4. Product-specific decay rates
5. Interaction features
6. Time-based features

Output: List of features to create with rationale based on your food science expertise.
"""

print("🔨 Data Scientist working on feature engineering...\n")

feature_plan = individual_meeting.run(
    agent=data_scientist,
    task=feature_task,
    critic=pi,  # PI provides feedback
    max_iterations=2
)

print("\n✅ Feature engineering plan completed!")
print("\n📝 Feature Plan:")
print(feature_plan)

## 7. Implement Features and Train Baseline Model

Based on the team's insights, we'll implement features and train a baseline model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

# Feature engineering based on agent insights
def create_features(batches_df, products_df, sites_df, regions_df):
    """Create features for shelf life prediction."""
    df = batches_df.copy()
    
    # Join reference data
    df = df.merge(products_df, on='sku_id', how='left')
    df = df.merge(sites_df, on='site_id', how='left')
    df = df.merge(regions_df, on='region_id', how='left')
    
    # Basic features
    features = [
        'dwell_hours',
        'mean_temp_F', 
        'mean_rh_pct',
        'door_opens_count',
        'base_shelf_life_days'
    ]
    
    # Engineered features
    # 1. Temperature abuse indicator (degree-days above optimal)
    df['temp_abuse'] = np.maximum(0, df['mean_temp_F'] - 40)  # 40°F as threshold
    df['thermal_load'] = df['temp_abuse'] * df['dwell_hours']
    
    # 2. Humidity stress
    df['humidity_stress'] = np.abs(df['mean_rh_pct'] - 85)  # 85% as optimal
    
    # 3. Door opening impact
    df['door_opens_per_hour'] = df['door_opens_count'] / (df['dwell_hours'] + 1)
    
    # 4. Remaining shelf life ratio
    df['shelf_life_consumption_rate'] = df['dwell_hours'] / (df['base_shelf_life_days'] * 24 + 1)
    
    # 5. Interaction features
    df['temp_humidity_interaction'] = df['mean_temp_F'] * df['mean_rh_pct']
    df['temp_doors_interaction'] = df['mean_temp_F'] * df['door_opens_count']
    
    # Add to features list
    features.extend([
        'temp_abuse', 'thermal_load', 'humidity_stress',
        'door_opens_per_hour', 'shelf_life_consumption_rate',
        'temp_humidity_interaction', 'temp_doors_interaction',
        'seasonality_amp'
    ])
    
    # Categorical features (one-hot encode)
    df = pd.get_dummies(df, columns=['category', 'storage_class', 'line_type'], prefix=['cat', 'storage', 'line'])
    
    # Get all categorical feature names
    cat_features = [col for col in df.columns if col.startswith(('cat_', 'storage_', 'line_'))]
    features.extend(cat_features)
    
    return df[features], df

# Create features
X_train, train_full = create_features(batches_train, products, sites, regions)
y_train = batches_train['shelf_life_remaining_days']

print(f"✅ Features created: {X_train.shape[1]} features, {X_train.shape[0]:,} samples")
print(f"\nFeature names: {list(X_train.columns[:10])}...")

# Handle missing values
X_train = X_train.fillna(0)

In [ ]:
# Train baseline model
print("🏋️ Training baseline Random Forest model...\n")

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Cross-validation
cv_scores = -cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

print(f"Cross-validation MAE: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"CV Scores: {cv_scores}")

# Train on full data
model.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 10 Most Important Features:")
print(feature_importance.head(10))

# Save model and results
baseline_results = {
    'model_type': 'RandomForest',
    'cv_mae_mean': float(cv_scores.mean()),
    'cv_mae_std': float(cv_scores.std()),
    'cv_scores': cv_scores.tolist(),
    'n_features': X_train.shape[1],
    'n_samples': X_train.shape[0],
    'timestamp': datetime.now().isoformat()
}

save_json(baseline_results, RESULTS_DIR / 'baseline_results.json')
print("\n✅ Baseline model trained and saved!")

## 8. Evolution Trigger Check

Check if performance plateau triggers evolution.

In [ ]:
# Simulate experiment history for trigger check
experiment_history = [
    {'mae': cv_scores.mean()},
]

# Create performance plateau trigger
plateau_trigger = PerformancePlateauTrigger(
    metric_name='mae',
    patience=3,
    min_improvement=0.1
)

should_evolve = plateau_trigger.should_trigger(experiment_history)

print(f"Performance Plateau Trigger: {should_evolve}")
print(f"Current MAE: {cv_scores.mean():.3f}")
print("\n💡 Continue iterating to build experiment history...")

## 9. Generate Test Predictions

Make predictions on the test set.

In [ ]:
# Prepare test data
X_test, test_full = create_features(batches_test, products, sites, regions)
X_test = X_test.fillna(0)

# Ensure same columns
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test[X_train.columns]

# Generate predictions
predictions = model.predict(X_test)

# Create submission
submission = pd.DataFrame({
    'batch_id': batches_test['batch_id'],
    'shelf_life_remaining_days': predictions
})

submission.to_csv(RESULTS_DIR / 'predictions_baseline.csv', index=False)

print(f"✅ Predictions generated for {len(predictions):,} test samples")
print(f"\n📊 Prediction Statistics:")
print(submission['shelf_life_remaining_days'].describe())
print(f"\n💾 Saved to: {RESULTS_DIR / 'predictions_baseline.csv'}")

## 10. Summary and Next Steps

Review what the Dream Team accomplished.

In [ ]:
print("="*60)
print("DREAM TEAM FRAMEWORK - SHELF LIFE PREDICTION EXPERIMENT")
print("="*60)

print("\n📊 RESULTS:")
print(f"  Baseline MAE (CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  Features engineered: {X_train.shape[1]}")
print(f"  Test predictions: {len(predictions):,}")

print("\n🧬 AGENT EVOLUTION:")
print(f"  Data Scientist:")
print(f"    Title: {data_scientist.title}")
print(f"    Specialization: {data_scientist.specialization_depth}")
print(f"    Knowledge: {len(data_scientist.knowledge_base.papers)} papers researched")

print("\n📁 ARTIFACTS SAVED:")
print(f"  - Agent snapshots: {RESULTS_DIR / 'agents'}")
print(f"  - Meeting transcripts: {RESULTS_DIR / 'meetings'}")
print(f"  - Predictions: {RESULTS_DIR / 'predictions_baseline.csv'}")
print(f"  - Results: {RESULTS_DIR / 'baseline_results.json'}")

print("\n🔄 NEXT STEPS:")
print("  1. Try different models (GradientBoosting, XGBoost)")
print("  2. Hyperparameter tuning")
print("  3. Research more papers if performance plateaus")
print("  4. Evolve ML Engineer with optimization techniques")
print("  5. Ensemble methods")
print("  6. Error analysis to identify evolution triggers")

print("\n✅ Experiment complete!")
print("="*60)